# APEX-seq (Fazal et al. 2019) vs unlabelled controls: read QC, nuclear RNA, compression and library complexity

Public APEX-seq data, processed with the bulk RNA-seq workflow of this project, as a reference for the ONT subcellular fractions of HCT116 (`ONT_organelles_vs_Total.ipynb`, same analyses). Fazal FM, Han S, Parker KR, et al. (2019) *Atlas of subcellular RNA localization revealed by APEX-Seq.* Cell 178:473–490, GEO [GSE116008](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116008).

| | |
|:---|:---|
| cells | HEK293T stably expressing APEX2 fused to a targeting sequence |
| constructs | NLS (nucleus), NIK (nucleolus), LMA (nuclear lamina), NucPore (nuclear pore), NES (cytosol), ERM (ER membrane, cytosolic face), KDEL (ER lumen), OMM (outer mitochondrial membrane), MITO (mitochondrial matrix) |
| libraries per construct | 2 labelled targets (biotin-phenol + H2O2, streptavidin enrichment), 2 unlabelled controls (no H2O2, otherwise identical, incl. the enrichment); MITO T2 has a technical replicate (added to T2 in the analyses with replicates) |
| fractionation | nuclear and cytosolic fractions (2 each, Gagnon et al. 2014), not enriched |
| library prep | Illumina TruSeq Stranded mRNA (poly(A)+), 2 × 75 nt, HiSeq 4000, ~35 M read pairs |
| not used | the drug (cycloheximide, puromycin, CCCP) and nocodazole perturbations of NES and OMM, and the total-RNA MITO libraries |

**Reference ("Total").** The 18 unlabelled controls together, as in the 18-control DESeq2 analysis of the paper: the RNA that binds to the beads without labelling, from the same cells. It plays the role of the Total RNA of the ONT data; the fractionation libraries are compared with the same reference although they were not enriched.

**Processing.** `WF/scripts/download_sra_fastq.sh` (SRA toolkit), the workflow `Snakefile-prepare-faster` (STAR, unique alignments), `Snakefile-quantification-faster` (featureCounts on GENCODE v42 basic, read pairs, reverse-stranded; PAQR coverages) and `Snakefile-PAQR-quantify` (run `wf_runs/APEX_seq_Fazal2019`, config as in `NFYA_project.ipynb`), then `WF/scripts/APEX_fracSanity_pipeline.py` (steps `prepare`, `complexity`).

**Expression scale: TPM.** Short reads come from along the whole fragmented transcript, so a gene's read count grows with the length of its mRNA. Expression is therefore length-corrected to TPM (read pairs / exonic length of the gene, the union of its exons as counted by featureCounts, renormalised to 10^6), which estimates the share of mRNA molecules. The ONT reads are full-length molecules without fragmentation (one read per molecule), so **log2 TPM here corresponds to log2 CPM of the ONT data**. The Sanity-based analyses (`APEX_seq.fracSanity.ipynb`) use the same length correction (`sanity_normalization(..., gene_lengths=...)`).

**Other differences to the ONT data.** There are no UMIs: library complexity is estimated from fragments with identical coordinates (both mates), which also counts natural duplicates of highly expressed genes.

Sections:
1. samples and read QC along the pipeline;
2. nuclear RNA, from nuclear marker RNAs;
3. densities of log2 TPM, target vs reference, and the principal-axis (PC1) slope;
4. sequencing depth: the slope expected from depth alone;
5. library complexity and its association with the slope;
6. control: APEX-seq against biochemical fractionation (nucleus vs cytosol).

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import csv

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr
from scipy.optimize import brentq

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR")}
ANALYSIS = "APEX_seq_vs_Control"
SU_OUT = os.path.join(subdirs["wf_runs_dir"], "Sanity_utils_APEX_seq_Fazal2019")   # WF/scripts/APEX_fracSanity_pipeline.py
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)
for d in (FIGS, TABLES):
    os.makedirs(d, exist_ok=True)

def figure_caption(path, formats=("png", "pdf")):
    # file name of a saved figure, shown with the plot; the links point to the full-size files relative to the
    # HTML report (WF/scripts/make_html_reports.py copies them to <report>/figures/, same subfolders as FIGURES_DIR)
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, name, formats=("png", "pdf")):
    for fmt in formats:
        fig.savefig(os.path.join(FIGS, f"{name}.{fmt}"), bbox_inches="tight", dpi=300)
    figure_caption(os.path.join(FIGS, name), formats)

def save_tsv(df, name, index=False):
    df.to_csv(os.path.join(TABLES, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)
    return os.path.join(TABLES, name)

# notation: colour = construct (fraction); marker = kind of library
FRACTION_ORDER = ["Total", "NLS", "NIK", "LMA", "NucPore", "NES", "ERM", "KDEL", "OMM", "MITO", "Nuc", "Cyto"]
FRACTION_COLORS = {"Total": "black", "NLS": "tab:red", "NIK": "darkred", "LMA": "tab:orange", "NucPore": "goldenrod", "NES": "tab:blue",
                   "ERM": "tab:green", "KDEL": "yellowgreen", "OMM": "tab:purple", "MITO": "tab:pink", "Nuc": "tab:brown", "Cyto": "tab:cyan"}
KIND_MARKERS = {"control": "D", "target": "o", "fractionation": "s"}

def legend_handles(df, col="fraction"):
    return [plt.Line2D([], [], color=FRACTION_COLORS[f], marker="o", ls="", label=f) for f in FRACTION_ORDER if (df[col] == f).any()] + \
           [plt.Line2D([], [], color="grey", marker=m, ls="", label=k) for k, m in KIND_MARKERS.items() if (df["kind"] == k).any()]

def color_ticklabels(labels, fractions):
    for t, f in zip(labels, fractions):
        t.set_color(FRACTION_COLORS.get(f, "black"))

## Samples

One row per sequenced library (`library`); `sample` is the unit of the analyses with replicates (MITO T2 and its technical replicate are one sample). Controls are labelled by the construct of the cells they come from.

In [ ]:
libs = pd.read_csv(os.path.join(SU_OUT, "inputs", "libraries.tsv"), sep="\t")
libs["fraction"] = libs["fraction"].where(libs["kind"] != "control", "Total")
order = {f: i for i, f in enumerate(FRACTION_ORDER)}
libs["construct_of_cells"] = np.where(libs["kind"] == "fractionation", "", libs["library"].str.split("-").str[1])
libs = libs.assign(_k=libs["kind"].map({"target": 0, "control": 1, "fractionation": 2}),
                   _f=libs["construct_of_cells"].map(lambda c: order.get(c, 99)), _g=libs["fraction"].map(order)) \
           .sort_values(["_k", "_f", "_g", "library"]).drop(columns=["_k", "_f", "_g"]).set_index("library", drop=False).rename_axis(None)
genes = pd.read_csv(os.path.join(SU_OUT, "inputs", "genes.tsv"), sep="\t", index_col=0)
gene_length = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_lengths.tsv"), sep="\t", index_col=0)["length"]   # exonic union, featureCounts
counts = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_counts.per_library.tsv.gz"), sep="\t", index_col=0)[libs.index]
save_tsv(libs, "libraries.tsv")
display(libs[["sample", "kind", "fraction", "construct_of_cells", "replicate", "gsm", "srr", "title"]])

control_libs = libs.index[libs["kind"] == "control"]
target_libs = libs.index[libs["kind"] == "target"]
frac_libs = libs.index[libs["kind"] == "fractionation"]
nuclear_pc = genes.index[(genes["gene_type"] == "protein_coding") & (genes["chrom"] != "chrM")]

## Read QC along the pipeline

Reads at the steps of the workflow (`mapping_stats/*.mapping_stats.txt`: input, mapped and uniquely mapped read pairs, as counted by STAR) and read pairs assigned to genes by featureCounts (GENCODE v42 basic, unique alignments, reverse-stranded). The composition of the assigned read pairs: nuclear protein-coding genes, mitochondrially encoded genes, lncRNAs, small RNAs (snoRNA, snRNA, scaRNA, misc_RNA, rRNA genes), other.

In [ ]:
qc = pd.read_csv(os.path.join(SU_OUT, "inputs", "read_qc.per_library.tsv"), sep="\t", index_col=0).reindex(libs.index)
qc["assigned read pairs"] = counts.sum()
qc["share uniquely mapped"] = qc["uniquely_mapped_reads"] / qc["input_reads"]
qc["share assigned (of uniquely mapped pairs)"] = qc["assigned read pairs"] / qc["uniquely_mapped_reads"]
SMALL = ["snoRNA", "snRNA", "scaRNA", "misc_RNA", "rRNA", "rRNA_pseudogene", "Mt_rRNA", "Mt_tRNA", "vault_RNA", "ribozyme", "sRNA", "scRNA"]
category = pd.Series("other", index=genes.index)
category[genes["gene_type"] == "lncRNA"] = "lncRNA"
category[genes["gene_type"].isin(SMALL)] = "small RNAs"
category[genes["gene_type"] == "protein_coding"] = "nuclear protein-coding"
category[genes["chrom"] == "chrM"] = "mitochondrially encoded"
CATS = ["nuclear protein-coding", "mitochondrially encoded", "lncRNA", "small RNAs", "other"]
CAT_COLORS = {"nuclear protein-coding": "tab:blue", "mitochondrially encoded": "tab:red", "lncRNA": "tab:orange", "small RNAs": "tab:purple", "other": "lightgrey"}
comp = counts.groupby(category.reindex(counts.index).to_numpy()).sum().reindex(CATS).fillna(0)
comp = (comp / comp.sum()).T
for c in CATS:
    qc[f"share {c}"] = comp[c]
save_tsv(libs[["sample", "kind", "fraction", "replicate"]].join(qc), "QC.read_stats_and_composition.tsv", index=True)
display(qc[["input_reads", "uniquely_mapped_reads", "share uniquely mapped", "assigned read pairs", "share assigned (of uniquely mapped pairs)"]
           + [f"share {c}" for c in CATS]].round(3))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 3, figsize=(16, 0.2 * len(libs) + 1.5), sharey=True, gridspec_kw={"width_ratios": [1.2, 1, 2]})
y = np.arange(len(libs))
ax = axes[0]
ax.barh(y, qc["input_reads"] / 1e6, color="lightgrey", height=0.8, label="input read pairs")
ax.barh(y, qc["uniquely_mapped_reads"] / 1e6, color="grey", height=0.8, label="uniquely mapped pairs")
ax.barh(y, qc["assigned read pairs"] / 1e6, color="black", height=0.8, label="assigned to genes")
ax.set(xlabel="million read pairs")
ax.legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=1)
ax = axes[1]
ax.barh(y, qc["share uniquely mapped"], color="grey", height=0.8)
ax.barh(y, qc["assigned read pairs"] / qc["input_reads"], color="black", height=0.8)
ax.set(xlabel="share of input: uniquely mapped (grey),\nassigned to genes (black)", xlim=(0, 1))
ax = axes[2]
left = np.zeros(len(libs))
for c in CATS:
    ax.barh(y, comp[c], left=left, color=CAT_COLORS[c], height=0.8, label=c)
    left += comp[c].to_numpy()
ax.set(xlim=(0, 1), xlabel="share of the read pairs assigned to genes")
ax.legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
axes[0].set_yticks(y, [f"{l} ({k[0]})" for l, k in zip(libs.index, libs["kind"])], fontsize=7)
axes[0].set_ylim(len(libs) - 0.5, -0.5)
color_ticklabels(axes[0].get_yticklabels(), libs["fraction"])
sns.despine(); fig.tight_layout()
savefig(fig, "read_qc.composition_per_library")
plt.show()

#### Interpretation: read QC

- **Depth:** 13–93 M read pairs per library, the fewest in MITO T1 (13 M). 70–94% are uniquely mapped; the lowest shares are MITO T2tech, KDEL C1 and NucPore T2 (70–71%).
- **Assignment to genes:**
  - 95–98% of the uniquely mapped pairs are assigned to exons in most libraries.
  - It is lower in the nuclear constructs (NLS 75–76%, LMA 72–79%, NIK 81–85%) and the nuclear fraction (81–82%). These carry more unspliced pre-mRNA, whose intronic reads are not counted.
  - One control (LMA C1) is also low, at 78%.
- **Composition:**
  - Nuclear protein-coding genes take 89–95% of the assigned pairs in every library except MITO, where the mitochondrially encoded genes take 46–62%.
  - The cytosolic fraction has 10–11% mitochondrial reads, against 2–5% in the controls.
  - lncRNAs are highest in NLS, LMA (4–5%) and the nuclear fraction (4%), against ~1.5% in the controls.

## Nuclear RNA

As in `ONT_organelles_vs_Total.ipynb`: for a nuclear marker RNA (MALAT1, NEAT1, snoRNAs), $R$ = (marker read pairs / read pairs on nuclear protein-coding genes) in the library, divided by the same ratio in the pooled 18 controls; 95% interval from the Poisson interval of the marker counts. In the ONT fractions, $R$ measured contamination with nuclear RNA; here it is expected to be high in the nuclear constructs (NLS, NIK, LMA, NucPore) and in the nuclear fraction, and low in the cytosolic and membrane constructs, which tests the specificity of the labelling.

In [ ]:
NUCLEAR_MARKERS = {"MALAT1": genes.index[genes["gene_name"] == "MALAT1"], "NEAT1": genes.index[genes["gene_name"] == "NEAT1"],
                   "snoRNAs": genes.index[genes["gene_type"] == "snoRNA"]}
nuc_reads = counts.reindex(nuclear_pc, fill_value=0).sum()
ref_nuc = nuc_reads[control_libs].sum()
nuc_rows = []
for m, ids in NUCLEAR_MARKERS.items():
    k = counts.reindex(ids, fill_value=0).sum()
    ref_rate = k[control_libs].sum() / ref_nuc
    lo = np.where(k > 0, stats.chi2.ppf(0.025, 2 * k) / 2, 0.0)
    hi = stats.chi2.ppf(0.975, 2 * k + 2) / 2
    nuc_rows.append(pd.DataFrame({"library": counts.columns, "marker": m, "marker_read_pairs": k.to_numpy(), "nuclear_pc_read_pairs": nuc_reads.to_numpy(),
                                  "R": (k / nuc_reads / ref_rate).to_numpy(), "R_low": lo / nuc_reads.to_numpy() / ref_rate,
                                  "R_high": hi / nuc_reads.to_numpy() / ref_rate}))
nuc_df = pd.concat(nuc_rows, ignore_index=True)
nuc_wide = nuc_df.pivot(index="library", columns="marker", values="R").reindex(libs.index)
for m in NUCLEAR_MARKERS:
    qc[f"R {m}"] = nuc_wide[m]
qc["log2 R MALAT1"] = np.log2(qc["R MALAT1"].clip(lower=1e-2))
save_tsv(nuc_df.merge(libs[["library", "sample", "kind", "fraction"]], on="library"), "nuclear_markers.per_library.tsv")

sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(13, 4.4))
x = np.arange(len(libs))
for j, (m, color) in enumerate(zip(NUCLEAR_MARKERS, ("tab:red", "tab:orange", "tab:purple"))):
    d = nuc_df.loc[nuc_df["marker"] == m].set_index("library").reindex(libs.index)
    yv = np.log2(d["R"].clip(lower=1e-2))
    ax.errorbar(x + (j - 1) * 0.22, yv, yerr=[yv - np.log2(d["R_low"].clip(lower=1e-2)), np.log2(d["R_high"].clip(lower=1e-2)) - yv],
                fmt="o", ms=3, lw=0.8, color=color, label=m)
ax.axhline(0, color="grey", lw=0.8, ls="--")
for k in ("control", "fractionation"):
    ax.axvline(np.where(libs["kind"] == k)[0][0] - 0.5, color="grey", lw=0.6)
ax.set_xticks(x, libs.index, rotation=90, fontsize=6.5)
color_ticklabels(ax.get_xticklabels(), libs["fraction"])
ax.set_ylabel("log2 R (relative to the 18 controls)")
ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "nuclear_markers_per_library")
plt.show()

gm = qc.join(libs[["kind", "fraction"]]).groupby(["kind", "fraction"])
nuc_group = np.exp(gm[["R MALAT1", "R NEAT1", "R snoRNAs"]].agg(lambda s: np.log(s.clip(lower=1e-2)).mean()))
nuc_group = nuc_group.reindex(sorted(nuc_group.index, key=lambda i: (["target", "control", "fractionation"].index(i[0]), order[i[1]])))
save_tsv(nuc_group.reset_index(), "nuclear_markers.by_fraction.tsv")
display(nuc_group.round(2))

#### Interpretation: nuclear RNA

- **The nuclear constructs enrich nuclear RNA 3–5-fold:** $R_{MALAT1}$ NLS 5.2, LMA 4.6, NIK 3.9, NucPore 3.4, with NEAT1 2.6–7.2. This is less than the biochemical nuclear fraction (MALAT1 8.5, NEAT1 12.3).
- **The cytosolic constructs deplete it about 2-fold** (NES 0.50, OMM 0.54), less than the cytosolic fraction (0.28). ERM is at the level of the controls (0.96). KDEL and MITO are slightly enriched (1.6), KDEL perhaps through the ER lumen continuous with the nuclear envelope.
- **APEX-seq separates nuclear from cytosolic RNA,** with a smaller dynamic range than fractionation. The 18 controls vary by up to 8-fold in their MALAT1 ratio (log2 $R$ −2.2 to +1.2 per library, table of the complexity section), which limits any per-library comparison against them.

## Expression: log2 TPM of nuclear protein-coding genes

Genes: nuclear protein-coding genes with ≥ 10 TPM in the pooled 18 controls; log2 TPM (read pairs + 0.5, divided by the exonic length of the gene, renormalised to 10^6 over these genes; the scale of the ONT log2 CPM), per library and for the replicates of a target (or fractionation) summed. Reference: the pooled controls. Below: densities of log2 TPM, target vs reference, and the scatter with the first principal axis of the centred (reference, target) values (PC1 slope; slope 1 = same spread as the reference, < 1 = compressed). The controls of each construct are also compared with the other 16 controls (leave the construct out), as the reference for the slope expected between libraries of the same kind.

In [ ]:
pooled_ctrl = counts[control_libs].sum(axis=1)
nuclear_pc = nuclear_pc[nuclear_pc.isin(gene_length.index)]
rate = pooled_ctrl[nuclear_pc] / gene_length[nuclear_pc]
tpm_ctrl = rate / rate.sum() * 1e6
universe = tpm_ctrl.index[tpm_ctrl >= 10]
print(f"genes: {len(universe):,} nuclear protein-coding genes with >= 10 TPM in the pooled controls "
      f"(of {len(nuclear_pc):,}; {pooled_ctrl[universe].sum() / pooled_ctrl[nuclear_pc].sum():.1%} of their read pairs)")

def log2_tpm(cnt, pseudocount=0.5):
    # length-corrected: (read pairs + pseudocount) / exonic length, renormalised over the genes of the universe
    x = (cnt.loc[universe] + pseudocount).div(gene_length.loc[universe], axis=0)
    return np.log2(x / x.sum() * 1e6)

def pc1_slope(t, p):
    # slope of the first principal axis of the centred (reference, fraction) values
    w, V = np.linalg.eigh(np.cov(np.vstack([t, p])))
    return V[1, -1] / V[0, -1]

qc["read pairs on genes"] = counts.loc[universe].sum()
t_ref = log2_tpm(pooled_ctrl).to_numpy()
lib_values = log2_tpm(counts)

# groups: replicates of a target / fractionation summed; controls of a construct summed
libs["group"] = np.where(libs["kind"] == "control", "Control " + libs["construct_of_cells"], libs["sample"].str.rsplit(" ", n=1).str[0])
group_libs = libs.groupby("group", sort=False)["library"].apply(list)
group_info = libs.groupby("group", sort=False)[["kind", "fraction", "construct_of_cells"]].first()
group_counts = pd.DataFrame({g: counts[l].sum(axis=1) for g, l in group_libs.items()})
group_values = log2_tpm(group_counts)

def slope_row(t, p):
    return {"pc1_slope": pc1_slope(t, p), "pearson_r": np.corrcoef(t, p)[0, 1], "sd_reference": t.std(), "sd_fraction": p.std(),
            "KS_fraction_vs_reference": stats.ks_2samp(p, t).statistic}

def reference_for(group):
    # controls of a construct are compared with the 16 other controls; everything else with all 18
    c = group_info.loc[group, "construct_of_cells"]
    if group_info.loc[group, "kind"] == "control":
        return log2_tpm(counts[[l for l in control_libs if libs.loc[l, "construct_of_cells"] != c]].sum(axis=1)).to_numpy()
    return t_ref

group_slopes = group_info.join(pd.DataFrame({g: {"n_libraries": len(group_libs[g]), "read_pairs_on_genes": int(group_counts[g].loc[universe].sum()),
                                                 **slope_row(reference_for(g), group_values[g].to_numpy())} for g in group_counts}).T)
lib_slopes = libs[["sample", "kind", "fraction", "group"]].join(pd.DataFrame({l: {"read_pairs_on_genes": int(qc.loc[l, "read pairs on genes"]),
                                                                                **slope_row(reference_for(libs.loc[l, "group"]), lib_values[l].to_numpy())}
                                                                            for l in libs.index}).T)
save_tsv(group_slopes.reset_index(names="group"), "fraction_vs_Control.pc1_slope.by_group.tsv")
save_tsv(lib_slopes.reset_index(names="library"), "fraction_vs_Control.pc1_slope.per_library.tsv")
display(group_slopes.round(3))

grid = np.linspace(-2, 14, 300)
lims = np.array([0.0, 14.0])
show = [g for g in group_info.index if group_info.loc[g, "kind"] != "control"]
ncol = 6
nrow = int(np.ceil(len(show) / ncol))
for kind_plot in ("density", "scatter"):
    sns.set(font_scale=0.85, style="ticks")
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.7 * ncol, 2.6 * nrow), sharex=True, sharey=True, squeeze=False)
    for ax, g in zip(axes.flat, show):
        f = group_info.loc[g, "fraction"]
        p = group_values[g].to_numpy()
        if kind_plot == "density":
            ax.plot(grid, stats.gaussian_kde(t_ref)(grid), color="black", lw=1.1, label="controls")
            ax.plot(grid, stats.gaussian_kde(p)(grid), color=FRACTION_COLORS[f], lw=1.7, label="target")
        else:
            ax.scatter(t_ref, p, s=1.2, alpha=0.25, color=FRACTION_COLORS[f], lw=0, rasterized=True)
            mt, mp, b = t_ref.mean(), p.mean(), pc1_slope(t_ref, p)
            ax.plot(lims, mp + (lims - mt), color="grey", lw=0.8, ls="--")
            ax.plot(lims, mp + b * (lims - mt), color="black", lw=1.2)
            ax.text(0.04, 0.96, f"PC1 slope {b:.2f}\nr {np.corrcoef(t_ref, p)[0, 1]:.2f}", transform=ax.transAxes, va="top", fontsize=7)
            ax.set(xlim=lims, ylim=lims, aspect="equal")
        ax.set_title(f"{g} (n={len(group_libs[g])})", fontsize=8, color=FRACTION_COLORS[f])
    for ax in axes.flat[len(show):]:
        ax.axis("off")
    for ax in axes[-1]:
        ax.set_xlabel("log2 TPM, controls" if kind_plot == "scatter" else "log2 TPM")
    for ax in axes[:, 0]:
        ax.set_ylabel("log2 TPM, target" if kind_plot == "scatter" else "density")
    if kind_plot == "density":
        axes[0, 0].legend(fontsize=7, frameon=False, loc="upper left")
    sns.despine(); fig.tight_layout()
    savefig(fig, f"fraction_vs_Control.{kind_plot}")
    plt.show()

## Sequencing depth: the slope expected from depth alone

Counting noise spreads the log2 TPM of shallow libraries, which can raise or lower the PC1 slope without any biology. Null: each control library subsampled (multivariate hypergeometric) to a range of depths and compared with the other 17 controls; and every target library against the pooled controls subsampled to the depth of the target (depth-matched slope). With ~10–30 M read pairs on these genes per library, the depth effect is expected to be small here, unlike in the ONT fractions.

In [ ]:
rng = np.random.default_rng(0)
counts_u = counts.loc[universe]
N_NULL, N_SUBSAMPLE = 5, 20
null_rows = []
depths = np.geomspace(1e5, 3e7, 12).astype(np.int64)
for l in control_libs[::3]:
    x = log2_tpm(counts[[r for r in control_libs if r != l]].sum(axis=1)).to_numpy()
    c = counts_u[l].to_numpy()
    for d in depths[depths < c.sum()]:
        for _ in range(N_NULL):
            sub = pd.Series(rng.multivariate_hypergeometric(c, d), index=universe)
            null_rows.append({"split": l, "depth": d, "pc1_slope": pc1_slope(x, log2_tpm(sub).to_numpy())})
null_df = pd.DataFrame(null_rows)
null_curve = null_df.groupby("depth")["pc1_slope"].agg(["mean", lambda s: s.quantile(0.025), lambda s: s.quantile(0.975)])
null_curve.columns = ["null_slope", "null_q025", "null_q975"]
save_tsv(null_curve.reset_index(), "depth_null.controls_subsampled_vs_controls.tsv")

def null_slope_at(depth):
    return np.interp(np.log10(depth), np.log10(null_curve.index), null_curve["null_slope"])

pooled_u = pooled_ctrl.loc[universe].to_numpy()
dm = {}
for l in libs.index[libs["kind"] != "control"]:
    p = lib_values[l].to_numpy()
    s = [pc1_slope(log2_tpm(pd.Series(rng.multivariate_hypergeometric(pooled_u, int(qc.loc[l, "read pairs on genes"])), index=universe)).to_numpy(), p)
         for _ in range(N_SUBSAMPLE)]
    dm[l] = np.mean(s)
lib_slopes["pc1_slope_depth_matched"] = pd.Series(dm)
lib_slopes["null_slope"] = null_slope_at(lib_slopes["read_pairs_on_genes"].astype(float))
lib_slopes["excess_compression"] = lib_slopes["pc1_slope"] - lib_slopes["null_slope"]
save_tsv(lib_slopes.reset_index(names="library"), "fraction_vs_Control.pc1_slope.per_library.tsv")
display(lib_slopes.loc[lib_slopes["kind"] != "control", ["fraction", "read_pairs_on_genes", "pc1_slope", "null_slope", "excess_compression",
                                                          "pc1_slope_depth_matched"]].astype({"pc1_slope": float}).round(3))

sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.fill_between(null_curve.index, null_curve["null_q025"], null_curve["null_q975"], color="lightgrey", lw=0)
ax.plot(null_curve.index, null_curve["null_slope"], color="grey", lw=1.2, label="control subsampled (null)")
for l, r in lib_slopes.iterrows():
    ax.scatter(r["read_pairs_on_genes"], r["pc1_slope"], color=FRACTION_COLORS[r["fraction"]], marker=KIND_MARKERS[r["kind"]], s=30,
               edgecolor="black", lw=0.3)
ax.axhline(1, color="grey", lw=0.8, ls="--")
ax.set(xscale="log", xlabel="read pairs of the library on the genes", ylabel="PC1 slope, library vs controls")
ax.legend(handles=legend_handles(lib_slopes), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "fraction_vs_Control.pc1_slope_vs_depth")
plt.show()

## Library complexity

Without UMIs, the duplicates are read pairs whose first mate starts at the same position, on the same strand, with the same insert size (uniquely mapped proper pairs; `APEX_fracSanity_pipeline.py --steps complexity`). With $N$ read pairs and $U$ distinct fragments, the Lander–Waterman estimate of the number of distinct fragments in the library, $C$, solves $U = C(1 - e^{-N/C})$, as in `ONT_organelles_vs_Total.ipynb`. In RNA-seq, highly expressed short genes produce identical fragments without PCR, so $C$ is a relative measure between libraries of one protocol. The slope of each target and fractionation library is then compared with its depth, complexity relative to the controls and nuclear RNA ($\log_2 R$ of MALAT1).

In [ ]:
def distinct_molecules(n_reads, n_unique):
    # Lander-Waterman: n_unique = C * (1 - exp(-n_reads / C)), solved for C
    if not (0 < n_unique < n_reads):
        return np.nan
    return brentq(lambda C: C * (1 - np.exp(-n_reads / C)) - n_unique, n_unique, n_unique * 1e6)

dup = pd.read_csv(os.path.join(SU_OUT, "inputs", "fragment_duplicates.per_library.tsv"), sep="\t", index_col=0).reindex(libs.index)
qc = qc.join(dup)
qc["share distinct fragments"] = qc["distinct_fragments"] / qc["read_pairs"]
qc["distinct molecules"] = [distinct_molecules(n, u) for n, u in zip(qc["read_pairs"], qc["distinct_fragments"])]
C_ctrl = np.exp(np.log(qc.loc[control_libs, "distinct molecules"]).mean())
qc["log2 complexity vs controls"] = np.log2(qc["distinct molecules"] / C_ctrl)
save_tsv(libs[["sample", "kind", "fraction", "replicate"]].join(qc), "QC.read_stats_composition_complexity.tsv", index=True)
display(qc[["read_pairs", "distinct_fragments", "share distinct fragments", "distinct molecules", "log2 complexity vs controls", "log2 R MALAT1"]].round(3))

LIB_X = {"read_pairs_on_genes": ("read pairs on the genes", "log"),
         "share distinct fragments": ("share of distinct fragments", "linear"),
         "log2 complexity vs controls": ("log2 distinct fragments (C),\nlibrary / controls", "linear"),
         "log2 R MALAT1": ("nuclear RNA, log2 R MALAT1", "linear")}
cplx = lib_slopes.loc[lib_slopes["kind"] != "control"].join(qc[["share distinct fragments", "log2 complexity vs controls", "log2 R MALAT1"]])
cplx[list(LIB_X)] = cplx[list(LIB_X)].astype(float)
cplx["pc1_slope"] = cplx["pc1_slope"].astype(float)
corr_rows = []
for k in LIB_X:
    r, p = spearmanr(cplx[k], cplx["pc1_slope"], nan_policy="omit")
    corr_rows.append({"x": k, "y": "pc1_slope", "n": len(cplx), "spearman": r, "p_value": p})
lib_corr = pd.DataFrame(corr_rows)
save_tsv(lib_corr, "fraction_vs_Control.pc1_slope.library_properties.spearman.tsv")
display(lib_corr.round(4))

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, len(LIB_X), figsize=(3.6 * len(LIB_X), 3.8), sharey=True)
for ax, (k, (xlabel, xscale)) in zip(axes, LIB_X.items()):
    for l, r in cplx.iterrows():
        ax.scatter(r[k], r["pc1_slope"], color=FRACTION_COLORS[r["fraction"]], marker=KIND_MARKERS[r["kind"]], s=30, edgecolor="black", lw=0.3)
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    c = lib_corr.set_index("x").loc[k]
    ax.set(xscale=xscale, xlabel=xlabel, title=f"Spearman {c['spearman']:.2f} (p = {c['p_value']:.2g})")
axes[0].set_ylabel("PC1 slope, library vs controls")
axes[-1].legend(handles=legend_handles(cplx), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "fraction_vs_Control.pc1_slope.library_properties")
plt.show()

## Pairwise comparisons: one library against one reference library

Above, every library is compared with the **pooled** controls: the counts of the 18 control libraries are summed, then converted to TPM (which sums to $10^6$ like any library; it is not an average of TPMs, and a constant offset over genes would not change a PC1 slope anyway). The pooled reference has much less sampling noise than one library, its libraries are weighted by their depth, and it mixes the cells of all nine constructs. Here each target library is compared with **the control libraries of its own construct** (the same cells, unlabelled): the control of the same replicate number ("matched"), and the mean over both controls of the construct. Fractionation libraries (no control of their own) are compared with each of the 18 controls (mean). Reference spread: each control against the other control of its construct (18 ordered pairs). The correlations with the library properties are repeated with the matched slopes.

In [ ]:
import re
def rep_number(lib):
    m = re.search(r"(\d)", lib.split("-")[-1])
    return int(m.group(1)) if m else np.nan

pair_rows = []
for l in libs.index:
    kind, c = libs.loc[l, "kind"], libs.loc[l, "construct_of_cells"]
    if kind == "target":
        refs = [r for r in control_libs if libs.loc[r, "construct_of_cells"] == c]
    elif kind == "control":
        refs = [r for r in control_libs if libs.loc[r, "construct_of_cells"] == c and r != l]
    else:
        refs = list(control_libs)
    for r in refs:
        pair_rows.append({"library": l, "kind": kind, "reference": r, "matched": kind != "fractionation" and rep_number(r) == rep_number(l) or kind == "control",
                          "pc1_slope": pc1_slope(lib_values[r].to_numpy(), lib_values[l].to_numpy())})
pairwise = pd.DataFrame(pair_rows)
save_tsv(pairwise, "fraction_vs_Control.pc1_slope.pairwise.tsv")
pw = pairwise.groupby("library").agg(pairwise_mean=("pc1_slope", "mean"), pairwise_min=("pc1_slope", "min"), pairwise_max=("pc1_slope", "max"))
pw["pairwise_matched"] = pairwise.loc[pairwise["matched"]].groupby("library")["pc1_slope"].mean()
pw["pairwise_matched"] = pw["pairwise_matched"].fillna(pw["pairwise_mean"])     # fractionation: mean over the controls
cmp_pw = lib_slopes[["kind", "fraction", "pc1_slope"]].astype({"pc1_slope": float}).join(pw)
cmp_pw["matched_minus_pooled"] = cmp_pw["pairwise_matched"] - cmp_pw["pc1_slope"]
save_tsv(cmp_pw.reset_index(names="library"), "fraction_vs_Control.pc1_slope.pairwise_vs_pooled.tsv")
display(cmp_pw.round(3))
cc = cmp_pw.loc[cmp_pw["kind"] == "control", "pairwise_matched"]
print(f"control vs the other control of its construct: slopes {cc.min():.3f}-{cc.max():.3f} (median {cc.median():.3f})")
nc = cmp_pw.loc[cmp_pw["kind"] != "control"]
print(f"targets and fractionation: matched - pooled slope median {nc['matched_minus_pooled'].median():+.3f} "
      f"(range {nc['matched_minus_pooled'].min():+.3f} to {nc['matched_minus_pooled'].max():+.3f}); "
      f"Spearman matched vs pooled {spearmanr(nc['pairwise_matched'], nc['pc1_slope'])[0]:.3f}")

cplx_pw = cplx.join(pw)
pw_rows = []
for k in LIB_X:
    r, p = spearmanr(cplx_pw[k], cplx_pw["pairwise_matched"], nan_policy="omit")
    pw_rows.append({"x": k, "y": "pairwise_matched", "n": len(cplx_pw), "spearman": r, "p_value": p})
pw_corr = pd.DataFrame(pw_rows)
save_tsv(pw_corr, "fraction_vs_Control.pc1_slope.pairwise.library_properties.spearman.tsv")
display(pd.concat({"pooled controls": lib_corr.set_index("x"), "matched control": pw_corr.set_index("x")})[["n", "spearman", "p_value"]].round(4))

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, 1 + len(LIB_X), figsize=(3.8 * (1 + len(LIB_X)), 3.9))
ax = axes[0]
for l, r in cmp_pw.iterrows():
    ax.errorbar(r["pc1_slope"], r["pairwise_matched"], yerr=[[max(r["pairwise_matched"] - r["pairwise_min"], 0)], [max(r["pairwise_max"] - r["pairwise_matched"], 0)]],
                fmt=KIND_MARKERS[r["kind"]], color=FRACTION_COLORS[r["fraction"]], ms=5, mec="black", mew=0.3, elinewidth=0.6, capsize=0)
lim = [min(cmp_pw[["pc1_slope", "pairwise_min"]].min()) * 0.95, max(cmp_pw[["pc1_slope", "pairwise_max"]].max()) * 1.05]
ax.plot(lim, lim, color="grey", ls="--", lw=0.8)
ax.set(xlim=lim, ylim=lim, xlabel="PC1 slope vs pooled controls", ylabel="PC1 slope vs matched control\n(range over the controls used)")
for ax, (k, (xlabel, xscale)) in zip(axes[1:], LIB_X.items()):
    for l, r in cplx_pw.iterrows():
        ax.scatter(r[k], r["pairwise_matched"], color=FRACTION_COLORS[r["fraction"]], marker=KIND_MARKERS[r["kind"]], s=30, edgecolor="black", lw=0.3)
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    c = pw_corr.set_index("x").loc[k]
    ax.set(xscale=xscale, xlabel=xlabel, ylabel="PC1 slope vs matched control", title=f"Spearman {c['spearman']:.2f} (p = {c['p_value']:.2g})")
axes[-1].legend(handles=legend_handles(cplx_pw), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "fraction_vs_Control.pc1_slope.pairwise")
plt.show()

#### Interpretation: pairwise comparisons

- **Single controls are much less consistent than in the ONT Total.** A control against the other control of its construct has slopes 0.62–1.62 (median 1.00): NES C1 and OMM C2 are much wider than their partners (1.61, 1.62), NucPore C1 1.22, NLS C1 1.10. Single-control slopes therefore depend on which control is used; for NES T1 and OMM T1/T2 the two controls give slopes that differ by 0.4.
- **Against the matched control (same construct and replicate number) the slopes are lower than against the pooled controls** (median −0.11, range −0.40 to 0.00), partly for the same reason as in the ONT data (one library adds noise on the reference axis) and partly because some single controls are themselves wide. The ordering is preserved (Spearman 0.93).
- **Complexity.** With the matched control, the slope correlates with the complexity of the library relative to the controls (Spearman −0.46, p = 0.03; pooled −0.36, p = 0.09): the least complex libraries (NIK T1, ERM T1) are *wider* than their controls (slopes 2.1–2.5), not compressed. Nuclear RNA (MALAT1) does not explain the slopes (−0.19).
- **Conclusion.** The pooled-control slopes are the more stable summary here; the pairwise analysis does not reveal a compression hidden by pooling. A pairwise analysis of complexity would need the complexity of the target relative to its own control (not to the mean of all controls), given how much single controls differ.

#### Interpretation: slopes, depth and complexity

- **The pooled targets are at most mildly compressed or expanded against the pooled controls.** PC1 slopes are 0.86–1.10 for NLS, LMA, NucPore, NES, ERM, KDEL, OMM and MITO; NLS and LMA (0.93, 0.86) are slightly compressed. NIK (1.35) and the cytosolic fraction (1.30) are expanded. The controls of one construct against the other 16 give 1.00–1.09.
- **Two libraries are strongly expanded: NIK T1 (2.74) and ERM T1 (2.25),** followed by NIK T2 (1.59). These are the libraries with the lowest complexity (1.7%, 2.0% and 2.4% distinct fragments). A library made from few molecules amplifies counting noise, which spreads the log2 TPM.
- **The depth null is above most observed slopes.** A single control library against the pooled controls gives a slope of 1.14–1.43, because the controls themselves are of low complexity (1.7–21% distinct fragments, except KDEL C1). The targets are less dispersed than that: slope minus null is −0.08 to −0.49, except the low-complexity libraries above.
- **Across the 23 target and fractionation libraries, no single variable explains the slope significantly.** Spearman with depth −0.12 (p 0.60), with complexity −0.36 (p 0.09), with MALAT1 $R$ −0.17 (p 0.44). The extreme slopes coincide with the lowest complexities.

## Control: APEX-seq against biochemical fractionation

The nuclear and cytosolic fractions (Gagnon et al. 2014 protocol, same cells and library protocol, no enrichment) give an independent measure of nuclear localization: per gene, $\log_2$(nucleus / cytosol) in TPM. If APEX-seq measures localization, $\log_2$(NLS / NES) of the targets should correlate with it, and the targets of the nuclear constructs (NIK, LMA, NucPore) should follow NLS. Below: Spearman correlations between $\log_2$(target / controls) of every construct and the fractionation $\log_2$(Nuc / Cyto), and the scatter for NLS / NES. Genes as above (≥ 10 TPM in the pooled controls).

In [ ]:
nuc_cyto = group_values["Fractionation: Nuc"] - group_values["Fractionation: Cyto"]
apex_groups = [g for g in group_info.index if group_info.loc[g, "kind"] == "target"]
enr = pd.DataFrame({g: group_values[g] - t_ref for g in apex_groups})
enr["NLS / NES"] = group_values["APEX: NLS"] - group_values["APEX: NES"]
cmp_rows = [{"APEX": g, "spearman_vs_log2_Nuc_Cyto": spearmanr(enr[g], nuc_cyto)[0], "n_genes": len(nuc_cyto)} for g in enr]
cmp_df = pd.DataFrame(cmp_rows)
save_tsv(cmp_df, "APEX_vs_fractionation.spearman.tsv")
save_tsv(pd.concat([enr, nuc_cyto.rename("log2 Nuc / Cyto")], axis=1).join(genes["gene_name"]).rename_axis("gene_id").reset_index(),
         "APEX_vs_fractionation.log2_ratios.per_gene.tsv.gz")
display(cmp_df.round(3))

sns.set(font_scale=0.9, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1, 1.2]})
ax = axes[0]
ax.scatter(nuc_cyto, enr["NLS / NES"], s=2, alpha=0.3, color="tab:red", lw=0, rasterized=True)
ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)
ax.set(xlabel="fractionation: log2 Nuc / Cyto", ylabel="APEX-seq: log2 NLS / NES",
       title=f"Spearman {spearmanr(nuc_cyto, enr['NLS / NES'])[0]:.2f}, {len(nuc_cyto):,} genes")
ax = axes[1]
d = cmp_df.set_index("APEX")["spearman_vs_log2_Nuc_Cyto"]
cols = [FRACTION_COLORS.get(g.split(": ")[-1], "grey") if ": " in g else "grey" for g in d.index]
ax.barh(np.arange(len(d)), d, color=cols, edgecolor="black", lw=0.3)
ax.set_yticks(np.arange(len(d)), d.index, fontsize=8)
ax.set_ylim(len(d) - 0.5, -0.5)
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="Spearman with fractionation log2 Nuc / Cyto")
sns.despine(); fig.tight_layout()
savefig(fig, "APEX_vs_fractionation")
plt.show()

#### Interpretation: APEX-seq and fractionation

- **The nuclear constructs agree best with fractionation:** Spearman of log2(construct / controls) with log2(Nuc / Cyto) 0.47 (NLS) and 0.50 (LMA).
- **However, every construct correlates positively,** including OMM (0.42), ERM and KDEL (0.35) and NES (0.15). The NLS/NES contrast correlates less (0.26) than NLS alone. A large part of the enrichment of any construct over the controls is therefore shared.
- **Part of the shared component is transcript length.** This was checked outside this notebook, with the exonic length from `inputs/gene_lengths.tsv`.
  - The enrichment correlates with the log2 exonic length for NES (0.53), ERM (0.60), LMA (0.46), NIK (0.45), OMM (0.43) and NLS (0.37), but not for KDEL, NucPore and MITO (−0.10 to 0.01). Longer RNAs carry more sites for the phenoxyl radical and are captured more efficiently.
  - log2(Nuc / Cyto) itself rises with length (0.28).
  - With length removed (partial rank correlation), NES loses its correlation with fractionation (0.005), NLS, LMA, KDEL, NucPore and OMM keep 0.35–0.43, and NLS / NES rises to 0.31.
- **APEX-seq recovers the nuclear–cytosolic axis, but only moderately** (≈ 0.3–0.5 rank correlation with fractionation), and its enrichments carry a length bias that a model of localization alone does not include.

## Summary

- **Libraries.** 41 libraries with 13–93 M read pairs, 70–94% uniquely mapped. The nuclear constructs and the nuclear fraction have the most intronic (unassigned) reads, and MITO is dominated by mitochondrial transcripts. The controls are of low complexity (mostly 2–10% distinct fragments), as are several targets (NIK, NucPore, ERM T1).
- **Nuclear RNA.** Enriched 3–5-fold by NLS, LMA, NIK and NucPore and depleted 2-fold by NES and OMM, a smaller range than fractionation (8.5 and 0.28).
- **Compression and expansion.** Small for the pooled constructs. The extreme slopes (NIK T1, ERM T1) are low-complexity artefacts.
- **Agreement with fractionation.** Moderate (Spearman ≤ 0.5), with a length-related component shared by all constructs. For the fracSanity analysis (`APEX_seq.fracSanity.ipynb`), the replicate disagreement of NIK and ERM and the low complexity of the controls are the main limitations.